## PHASE 3.0 - DATA PREPARATION
# TASK 3.1 - SELECTING DATA

In [1]:
#Importing Python Libaries
import pandas as pd
import numpy as np, json

## LendingClub Dataset
#### Link to dataset: 
https://zenodo.org/records/11295916?

In [2]:
#Loading the LendingClub data
df = pd.read_csv(r"C:\Users\moyos\Downloads\Lendingclub.csv")
print(f"Shape: {df.shape}")
df.head()

C:\Users\moyos\AppData\Local\Temp\ipykernel_1644\3982024170.py:2: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\moyos\Downloads\Lendingclub.csv")


Shape: (1347681, 15)


,id,issue_d,revenue,dti_n,loan_amnt,fico_n,experience_c,emp_length,purpose,home_ownership_n,addr_state,zip_code,Default,title,desc
0,68407277,Dec-2015,55000.0,5.91,3600,677.0,1,10+ years,debt_consolidation,MORTGAGE,PA,190xx,0,Debt consolidation,NaN
1,68355089,Dec-2015,65000.0,16.06,24700,717.0,1,10+ years,small_business,MORTGAGE,SD,577xx,0,Business,NaN
2,68341763,Dec-2015,71000.0,13.85,20000,697.0,1,10+ years,home_improvement,MORTGAGE,IL,605xx,0,NaN,NaN
3,68476807,Dec-2015,104433.0,25.37,10400,697.0,1,3 years,major_purchase,MORTGAGE,PA,174xx,0,Major purchase,NaN
4,68426831,Dec-2015,34000.0,10.20,11950,692.0,1,4 years,debt_consolidation,RENT,GA,300xx,0,Debt consolidation,NaN


In [3]:
def rate_by(df, col, target='Default'):
    return df.groupby(col)[target].agg(['count', 'mean']).round(4)

def decile_rate(df, col, target='Default', q=10):
    return df.groupby(pd.qcut(df[col], q, duplicates='drop'), observed=True)[target].mean().round(4)

df['year'] = pd.to_datetime(df['issue_d'], format='mixed').dt.year

In [4]:
# 1.Understanding if "id" is a time proxy
print('id vs issue_d (Spearman):', round(pd.to_numeric(df['id'], errors='coerce')
      .corr(pd.to_datetime(df['issue_d'], format='mixed').astype('int64'), method='spearman'), 4))

id vs issue_d (Spearman): 0.9986


In [5]:
# 2. Numeric predictors: default rate by decile
for c in ['revenue','loan_amnt','fico_n']:
    print(c); print(decile_rate(df, c), '\n')
print('dti_n (999 excluded)'); print(decile_rate(df[df['dti_n'] < 999], 'dti_n'))
print('dti_n > 100:', (df['dti_n'] > 100).sum(), '| default rate:', round(df.loc[df['dti_n'] > 100, 'Default'].mean(), 4))
print('dti_n == 999:', (df['dti_n'] == 999).sum())

revenue
revenue
(1895.999, 35000.0]       0.2377
(35000.0, 42675.0]        0.2273
(42675.0, 50000.0]        0.2183
(50000.0, 58000.0]        0.2106
(58000.0, 65000.0]        0.2096
(65000.0, 75000.0]        0.1976
(75000.0, 85000.0]        0.1904
(85000.0, 100000.0]       0.1798
(100000.0, 129000.0]      0.1670
(129000.0, 10999200.0]    0.1535
Name: Default, dtype: float64 

loan_amnt
loan_amnt
(499.999, 5000.0]     0.1581
(5000.0, 6500.0]      0.1593
(6500.0, 8625.0]      0.1691
(8625.0, 10000.0]     0.1828
(10000.0, 12000.0]    0.2049
(12000.0, 15000.0]    0.2102
(15000.0, 18000.0]    0.2274
(18000.0, 21000.0]    0.2267
(21000.0, 28000.0]    0.2250
(28000.0, 40000.0]    0.2448
Name: Default, dtype: float64 

fico_n
fico_n
(611.999, 667.0]    0.2630
(667.0, 672.0]      0.2462
(672.0, 677.0]      0.2379
(677.0, 682.0]      0.2309
(682.0, 692.0]      0.2116
(692.0, 697.0]      0.1994
(697.0, 707.0]      0.1828
(707.0, 722.0]      0.1564
(722.0, 742.0]      0.1313
(742.0, 847.5]      0.0

In [6]:
# 3. Overlap among numeric predictors
print(df[['loan_amnt','revenue','dti_n','fico_n']].corr('spearman').round(3))

           loan_amnt  revenue  dti_n  fico_n
loan_amnt      1.000    0.493  0.046   0.117
revenue        0.493    1.000 -0.186   0.110
dti_n          0.046   -0.186  1.000  -0.040
fico_n         0.117    0.110 -0.040   1.000


In [7]:
# 4. Categorical predictors
print(rate_by(df, 'emp_length').sort_values('mean'))
print(rate_by(df, 'purpose').sort_values('count', ascending=False))
print(rate_by(df, 'home_ownership_n').sort_values('mean'))

             count    mean
emp_length                
10+ years   442667  0.1880
6 years      62877  0.1938
7 years      59724  0.1951
5 years      84326  0.1962
4 years      80761  0.1976
2 years     122091  0.1982
9 years      51019  0.1991
8 years      60808  0.1995
3 years     107862  0.1999
< 1 year    108517  0.2054
1 year       88841  0.2059
NI           78188  0.2696
                     count    mean
purpose                           
debt_consolidation  781206  0.2115
credit_card         295551  0.1693
home_improvement     87684  0.1776
other                78263  0.2108
major_purchase       29543  0.1860
medical              15606  0.2184
small_business       15575  0.2986
car                  14647  0.1470
moving                9520  0.2341
vacation              9083  0.1919
house                 7294  0.2191
wedding               2350  0.1243
renewable_energy       936  0.2372
educational            423  0.2080
                   count    mean
home_ownership_n             

In [8]:
# 5. Time effects: NI share and category mix by year
print(df.groupby('year')['emp_length'].apply(lambda s: (s == 'NI').mean()).round(3))
print(pd.crosstab(df['year'], df['home_ownership_n'], normalize='index').round(3))
print(pd.crosstab(df['year'], df['purpose'], normalize='index').round(3))
print(df.groupby('year')['desc'].apply(lambda s: s.notna().mean()).round(3))

year
2007    0.000
2008    0.000
2009    0.000
2010    0.028
2011    0.035
2012    0.036
2013    0.044
2014    0.052
2015    0.059
2016    0.064
2017    0.071
2018    0.085
Name: emp_length, dtype: float64
home_ownership_n  MORTGAGE  OTHER    OWN   RENT
year                                           
2007                 0.344  0.005  0.087  0.564
2008                 0.389  0.020  0.062  0.529
2009                 0.388  0.016  0.089  0.507
2010                 0.448  0.000  0.077  0.475
2011                 0.468  0.000  0.074  0.458
2012                 0.467  0.002  0.079  0.452
2013                 0.535  0.000  0.083  0.382
2014                 0.505  0.000  0.098  0.397
2015                 0.485  0.000  0.109  0.406
2016                 0.487  0.000  0.122  0.391
2017                 0.501  0.001  0.120  0.378
2018                 0.506  0.000  0.134  0.360
purpose    car  credit_card  debt_consolidation  educational  \
year                                                      

In [9]:
# 6. experience_c, title, desc
print(df['experience_c'].value_counts()); print(rate_by(df, 'experience_c'))
print(df['title'].nunique(), df['title'].str.lower().value_counts().head(15))
print(rate_by(df.assign(title_missing=df['title'].isna()), 'title_missing'))
print(rate_by(df.assign(desc_present=df['desc'].notna()), 'desc_present'))

experience_c
1    1347656
0         25
Name: count, dtype: int64
                count    mean
experience_c                 
0                  25  0.1200
1             1347656  0.1998
61451 title
debt consolidation           683273
credit card refinancing      250260
home improvement              77795
other                         66839
major purchase                24353
medical expenses              13686
business                      11869
car financing                 11247
vacation                       8255
consolidation                  8141
moving and relocation          7947
home buying                    5912
debt consolidation loan        4824
credit card consolidation      3661
personal loan                  3418
Name: count, dtype: int64
                 count    mean
title_missing                 
False          1331024  0.1987
True             16657  0.2900
                count    mean
desc_present                 
False         1228582  0.2043
True           119099  

In [10]:
# 7. Geography
s = rate_by(df, 'addr_state')
print(df['addr_state'].nunique(), s.sort_values('count').head(10), s.sort_values('count').tail(10))
z = rate_by(df, 'zip_code')
print(df['zip_code'].head(), z['count'].describe(), (z['count'] < 100).sum())
pairs = df.groupby(['zip_code','addr_state']).size()
print('share in prefix main state:', round(pairs.groupby('zip_code').max().sum() / len(df), 4))

51             count    mean
addr_state               
IA             14  0.2143
ND           1601  0.2049
ID           1689  0.1883
ME           2029  0.1385
VT           2655  0.1397
SD           2770  0.2134
WY           2926  0.1675
AK           3194  0.1966
DC           3485  0.1323
NE           3591  0.2520              count    mean
addr_state                
VA           38107  0.1995
GA           43470  0.1843
OH           43935  0.2052
PA           45640  0.2082
NJ           48576  0.2112
IL           51858  0.1811
FL           95815  0.2150
NY          110075  0.2205
TX          110307  0.1984
CA          196805  0.1963
0    190xx
1    577xx
2    605xx
3    174xx
4    300xx
Name: zip_code, dtype: object count      945.000000
mean      1426.116402
std       2009.994493
min          1.000000
25%        269.000000
50%        669.000000
75%       1715.000000
max      15013.000000
Name: count, dtype: float64 129
share in prefix main state: 0.9985


In [11]:
# 8. Role assignment
lc_roles = {
    'primary': ['revenue','dti_n','loan_amnt','fico_n','emp_length','purpose','home_ownership_n'],
    'target': ['Default'], 'identifier': ['id'], 'time_validation': ['issue_d'],
    'audit': ['addr_state','zip_code'], 'excluded': ['experience_c','title','desc'],
}
allc = [c for v in lc_roles.values() for c in v]
assert len(allc) == len(set(allc)) == 15 == len([c for c in df.columns if c != 'year'])
json.dump(lc_roles, open('lc_roles.json', 'w'), indent=2)

## Home Credit (application_train.csv) Dataset
#### Link to dataset: 
https://www.kaggle.com/competitions/home-credit-default-risk/data?select=application_train.csv

In [12]:
#Loading Home Credit (application_train.csv) Dataset
df_HC= pd.read_csv(r"C:\Users\moyos\Downloads\HomeCredit.csv")
print(f"Shape: {df_HC.shape}")
df_HC.head()

Shape: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
from sklearn.metrics import roc_auc_score
y = df_HC['TARGET']
NOT_EMPLOYED = df_HC['DAYS_EMPLOYED'] == 365243

def rate_by(df, col, target='TARGET'):
    """Count and target rate per category."""
    return df.groupby(col)[target].agg(['count', 'mean']).round(4)

### 1. Screen (supports 3.1 group decisions)

In [14]:
# 1.Screen: Evualting missingness, AUC (numeric), rate spread (categorical)
def screen_columns(df, target='TARGET'):
    out = []
    for c in df.columns.drop(target):
        s = df[c]
        row = {'var': c, 'missing_%': round(s.isna().mean()*100, 2), 'nunique': s.nunique()}
        if s.dtype == 'object':
            r = df.groupby(s)[target].mean()
            row['rate_spread_pts'] = round((r.max() - r.min())*100, 2)
        else:
            row['auc'] = round(roc_auc_score(df[target], s.fillna(s.median())), 3)
        out.append(row)
    return pd.DataFrame(out).sort_values('missing_%', ascending=False)

screen = screen_columns(df_HC)
screen.to_csv('hc_screen.csv', index=False)
print(screen.to_string(index=False))

                         var  missing_%  nunique   auc  rate_spread_pts
              COMMONAREA_AVG      69.87     3181 0.493              NaN
             COMMONAREA_MEDI      69.87     3202 0.493              NaN
             COMMONAREA_MODE      69.87     3128 0.494              NaN
     NONLIVINGAPARTMENTS_AVG      69.43      386 0.489              NaN
    NONLIVINGAPARTMENTS_MEDI      69.43      214 0.489              NaN
    NONLIVINGAPARTMENTS_MODE      69.43      167 0.491              NaN
          FONDKAPREMONT_MODE      68.39        4   NaN             1.72
       LIVINGAPARTMENTS_MEDI      68.35     1097 0.488              NaN
        LIVINGAPARTMENTS_AVG      68.35     1868 0.489              NaN
       LIVINGAPARTMENTS_MODE      68.35      736 0.489              NaN
              FLOORSMIN_MODE      67.85       25 0.488              NaN
               FLOORSMIN_AVG      67.85      305 0.487              NaN
              FLOORSMIN_MEDI      67.85       47 0.487          

In [15]:
# 2.Loan terms: understanding redundancy
print(df_HC[['AMT_CREDIT','AMT_GOODS_PRICE','AMT_ANNUITY','AMT_INCOME_TOTAL']].corr('spearman').round(3))

                  AMT_CREDIT  AMT_GOODS_PRICE  AMT_ANNUITY  AMT_INCOME_TOTAL
AMT_CREDIT             1.000            0.985        0.830             0.412
AMT_GOODS_PRICE        0.985            1.000        0.828             0.416
AMT_ANNUITY            0.830            0.828        1.000             0.482
AMT_INCOME_TOTAL       0.412            0.416        0.482             1.000


### 3. AMT_GOODS_PRICE vs AMT_CREDIT (supports exclusion of raw column)

In [16]:
# 3.Viewing AMT_GOODS_PRICE vs AMT_CREDIT (the missing check)
d = df_HC['AMT_CREDIT'] - df_HC['AMT_GOODS_PRICE']
print('identical share:', (d == 0).mean().round(3))
print('AUC of difference:', round(roc_auc_score(y[d.notna()], d.dropna()), 3))
print(rate_by(df_HC.assign(gp_missing=df_HC['AMT_GOODS_PRICE'].isna()), 'gp_missing'))

identical share: 0.352
AUC of difference: 0.544
             count    mean
gp_missing                
False       307233  0.0807
True           278  0.0755


### 4. DAYS_EMPLOYED placeholder and XNA organization (supports 3.2 cleaning)

In [17]:
# 4.DAYS_EMPLOYED placeholder, XNA organization, occupation
print('placeholder rows:', NOT_EMPLOYED.sum())
print(pd.crosstab(NOT_EMPLOYED, df_HC['NAME_INCOME_TYPE']))
print(pd.crosstab(df_HC['ORGANIZATION_TYPE'] == 'XNA', NOT_EMPLOYED))
occ_na = df_HC['OCCUPATION_TYPE'].isna()
print(pd.crosstab(occ_na, df_HC['NAME_INCOME_TYPE']))
print('missing occupation, non-pensioners:',
      round(y[occ_na & (df_HC['NAME_INCOME_TYPE'] != 'Pensioner')].mean(), 4))
ok = ~NOT_EMPLOYED
print('DAYS_EMPLOYED AUC without placeholder:', round(roc_auc_score(y[ok], df_HC.loc[ok, 'DAYS_EMPLOYED']), 3))
print('DAYS_EMPLOYED vs age:', round(df_HC.loc[ok, 'DAYS_EMPLOYED'].corr(df_HC.loc[ok, 'DAYS_BIRTH'], method='spearman'), 3))

placeholder rows: 55374
NAME_INCOME_TYPE  Businessman  Commercial associate  Maternity leave  \
DAYS_EMPLOYED                                                          
False                      10                 71617                5   
True                        0                     0                0   

NAME_INCOME_TYPE  Pensioner  State servant  Student  Unemployed  Working  
DAYS_EMPLOYED                                                             
False                    10          21703       18           0   158774  
True                  55352              0        0          22        0  
DAYS_EMPLOYED       False  True 
ORGANIZATION_TYPE               
False              252137      0
True                    0  55374
NAME_INCOME_TYPE  Businessman  Commercial associate  Maternity leave  \
OCCUPATION_TYPE                                                        
False                       8                 59320                4   
True                        2          

### 5. EXT_SOURCE overlap, gender and age (supports EXT_SOURCE_1 exclusion)

In [18]:
# 5.EXT_SOURCE: overlap, gender, age, missingness
ext = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
print(df_HC[ext].corr('spearman').round(3))
print(df_HC.groupby('CODE_GENDER')[ext].mean().round(3))
print(df_HC[ext].corrwith(df_HC['DAYS_BIRTH'], method='spearman').round(3))
print(df_HC.groupby('CODE_GENDER')['EXT_SOURCE_1'].apply(lambda s: s.isna().mean()).round(3))

              EXT_SOURCE_1  EXT_SOURCE_2  EXT_SOURCE_3
EXT_SOURCE_1         1.000         0.221         0.181
EXT_SOURCE_2         0.221         1.000         0.106
EXT_SOURCE_3         0.181         0.106         1.000
             EXT_SOURCE_1  EXT_SOURCE_2  EXT_SOURCE_3
CODE_GENDER                                          
F                   0.546         0.516         0.514
M                   0.407         0.511         0.505
XNA                 0.529         0.581         0.302
EXT_SOURCE_1   -0.600
EXT_SOURCE_2   -0.102
EXT_SOURCE_3   -0.205
dtype: float64
CODE_GENDER
F      0.547
M      0.596
XNA    0.500
Name: EXT_SOURCE_1, dtype: float64


### 6. Housing block (supports exclusion of 47 columns)

In [19]:
# 6.Housing block and small groups
hb = [c for c in df_HC.columns if c.endswith(('_AVG','_MODE','_MEDI'))]
print(len(hb)); print(rate_by(df_HC.assign(block_missing=df_HC[hb].isna().all(axis=1)), 'block_missing'))
print(rate_by(df_HC, 'NAME_INCOME_TYPE').sort_values('count'))

47
                count    mean
block_missing                
False          161756  0.0700
True           145755  0.0926
                       count    mean
NAME_INCOME_TYPE                    
Maternity leave            5  0.4000
Businessman               10  0.0000
Student                   18  0.0000
Unemployed                22  0.3636
State servant          21703  0.0575
Pensioner              55362  0.0539
Commercial associate   71617  0.0748
Working               158774  0.0959


In [20]:
# 7.Rates by category quoted in the report
for c in ['NAME_CONTRACT_TYPE','CODE_GENDER','NAME_EDUCATION_TYPE','NAME_FAMILY_STATUS','NAME_HOUSING_TYPE']:
    print(rate_by(df_HC, c), '\n')

                     count    mean
NAME_CONTRACT_TYPE                
Cash loans          278232  0.0835
Revolving loans      29279  0.0548 

              count    mean
CODE_GENDER                
F            202448  0.0700
M            105059  0.1014
XNA               4  0.0000 

                                count    mean
NAME_EDUCATION_TYPE                          
Academic degree                   164  0.0183
Higher education                74863  0.0536
Incomplete higher               10277  0.0848
Lower secondary                  3816  0.1093
Secondary / secondary special  218391  0.0894 

                       count    mean
NAME_FAMILY_STATUS                  
Civil marriage         29775  0.0994
Married               196432  0.0756
Separated              19770  0.0819
Single / not married   45444  0.0981
Unknown                    2  0.0000
Widow                  16088  0.0582 

                      count    mean
NAME_HOUSING_TYPE                  
Co-op apartment       

In [22]:
import json
hc_roles = {
    'primary': ['NAME_CONTRACT_TYPE','AMT_CREDIT','AMT_ANNUITY','AMT_INCOME_TOTAL','NAME_INCOME_TYPE',
                'OCCUPATION_TYPE','ORGANIZATION_TYPE','DAYS_EMPLOYED','NAME_EDUCATION_TYPE',
                'NAME_HOUSING_TYPE','EXT_SOURCE_2','EXT_SOURCE_3'],
    'target': ['TARGET'], 'identifier': ['SK_ID_CURR'],
    'audit': ['CODE_GENDER','DAYS_BIRTH','NAME_FAMILY_STATUS','REGION_POPULATION_RELATIVE',
              'REGION_RATING_CLIENT','REGION_RATING_CLIENT_W_CITY'],
}
excluded_groups = {
    'document_flags': [f'FLAG_DOCUMENT_{i}' for i in range(2, 22)],
    'contact_flags': ['FLAG_MOBIL','FLAG_EMP_PHONE','FLAG_WORK_PHONE','FLAG_CONT_MOBILE','FLAG_PHONE','FLAG_EMAIL'],
    'address_flags': ['REG_REGION_NOT_LIVE_REGION','REG_REGION_NOT_WORK_REGION','LIVE_REGION_NOT_WORK_REGION',
                      'REG_CITY_NOT_LIVE_CITY','REG_CITY_NOT_WORK_CITY','LIVE_CITY_NOT_WORK_CITY'],
    'bureau': [c for c in df_HC.columns if c.startswith('AMT_REQ_CREDIT_BUREAU')],
    'social': ['OBS_30_CNT_SOCIAL_CIRCLE','DEF_30_CNT_SOCIAL_CIRCLE','OBS_60_CNT_SOCIAL_CIRCLE','DEF_60_CNT_SOCIAL_CIRCLE'],
    'process': ['WEEKDAY_APPR_PROCESS_START','HOUR_APPR_PROCESS_START','DAYS_ID_PUBLISH',
                'DAYS_REGISTRATION','DAYS_LAST_PHONE_CHANGE','NAME_TYPE_SUITE'],
    'household_assets': ['CNT_CHILDREN','CNT_FAM_MEMBERS','FLAG_OWN_CAR','FLAG_OWN_REALTY','OWN_CAR_AGE'],
    'other': ['EXT_SOURCE_1','AMT_GOODS_PRICE'],
    'building_block': [c for c in df_HC.columns if c.endswith(('_AVG','_MODE','_MEDI'))],
}
hc_roles['excluded'] = [c for v in excluded_groups.values() for c in v]

allc = [c for v in hc_roles.values() for c in v]
assert len(allc) == len(set(allc)) == df_HC.shape[1], 'roles do not cover each column exactly once'
json.dump(hc_roles, open('hc_roles.json', 'w'), indent=2)
print({k: len(v) for k, v in hc_roles.items()})

{'primary': 12, 'target': 1, 'identifier': 1, 'audit': 6, 'excluded': 102}


In [23]:
import json

expected = {
    'lc': {'primary': 7, 'target': 1, 'identifier': 1, 'time_validation': 1, 'audit': 2, 'excluded': 3},
    'hc': {'primary': 12, 'target': 1, 'identifier': 1, 'audit': 6, 'excluded': 102},
}
lc_roles = json.load(open('lc_roles.json'))
hc_roles = json.load(open('hc_roles.json'))

for name, roles, exp, total in [('LendingClub', lc_roles, expected['lc'], 15),
                                ('Home Credit', hc_roles, expected['hc'], 122)]:
    got = {k: len(v) for k, v in roles.items()}
    print(name, got)
    assert got == exp, f'{name}: counts differ from the paper'
    assert sum(got.values()) == total

# Build the selected datasets (everything except excluded)
keep_lc = [c for k, v in lc_roles.items() if k != 'excluded' for c in v]
keep_hc = [c for k, v in hc_roles.items() if k != 'excluded' for c in v]
print(len(keep_lc), len(keep_hc))        # expect 12 and 20
assert len(keep_lc) == 12 and len(keep_hc) == 20

# AMT_GOODS_PRICE is excluded as a raw predictor but needed in 3.3 to build credit-minus-goods-price
keep_hc_for_prep = keep_hc + ['AMT_GOODS_PRICE']

LendingClub {'primary': 7, 'target': 1, 'identifier': 1, 'time_validation': 1, 'audit': 2, 'excluded': 3}
Home Credit {'primary': 12, 'target': 1, 'identifier': 1, 'audit': 6, 'excluded': 102}
12 20
